In [3]:
import pandas as pd
import glob
import sys

1. CONCATENAR LOS DATASETS Y VERIFICAR QUE TODO ESTE EN ORDEN

In [4]:
archivos = ['Data_Set/TradeData_10_13_2026_4_37_2.csv', 'Data_Set/TradeData_10_13_2026_4_38_34.csv']
dfs = [pd.read_csv(f) for f in archivos]
df = pd.concat(dfs, ignore_index=True)
print(f"1. Filas iniciales: {len(df)}")
df.head()

1. Filas iniciales: 153476


,typeCode,freqCode,refPeriodId,refYear,refMonth,period,reporterCode,reporterISO,reporterDesc,flowCode,...,netWgt,isNetWgtEstimated,grossWgt,isGrossWgtEstimated,cifvalue,fobvalue,primaryValue,legacyEstimationFlag,isReported,isAggregate
0,A,20250101,2025,52,2025,20,AND,Andorra,M,Import,...,True,0,False,1390332.194,NaN,1390332.194,4,False,True,NaN
1,A,20250101,2025,52,2025,20,AND,Andorra,M,Import,...,False,0,False,35995.814,NaN,35995.814,0,False,True,NaN
2,A,20250101,2025,52,2025,20,AND,Andorra,M,Import,...,False,0,False,184.427,NaN,184.427,0,False,True,NaN
3,A,20250101,2025,52,2025,20,AND,Andorra,M,Import,...,False,0,False,74.859,NaN,74.859,0,False,True,NaN
4,A,20250101,2025,52,2025,20,AND,Andorra,M,Import,...,False,0,False,5597.810,NaN,5597.810,0,False,True,NaN


PASO 2. RETIRAR AQUELLOS QUE NO SON PAISES.

In [5]:
# Limpiar espacios en nombres de columnas
df.columns = df.columns.str.strip()

# --- CORRECCIÓN DEL DESPLAZAMIENTO DE COLUMNAS ---
# Verificamos si la columna 'flowCode' contiene los textos de 'flowDesc'
if df['flowCode'].astype(str).str.strip().str.title().isin(['Import', 'Export']).any():
    print("Desplazamiento de columnas detectado. Reestructurando cabeceras...")
    # Desplazamos los nombres de las columnas una posición a la izquierda
    nuevas_columnas = list(df.columns[1:]) + ['columna_extra']
    df.columns = nuevas_columnas
# -------------------------------------------------

print(f"1. Total de filas iniciales: {len(df)}")

Desplazamiento de columnas detectado. Reestructurando cabeceras...
1. Total de filas iniciales: 153476


In [6]:
# 2. Filtrar el flujo comercial (Importaciones y Exportaciones)
df_limpio = df[df['flowDesc'].isin(['Import', 'Export'])].copy()
print(f"2. Filas tras filtrar solo 'Import' o 'Export': {len(df_limpio)}")

2. Filas tras filtrar solo 'Import' o 'Export': 153476


In [7]:
# 3. Eliminar el agregado mundial y regiones no soberanas para tener solo red bilateral
df_limpio['partnerDesc'] = df_limpio['partnerDesc'].astype(str).str.strip()
df_limpio['reporterDesc'] = df_limpio['reporterDesc'].astype(str).str.strip()

# Lista de palabras clave que identifican agregados en Comtrade
terminos_excluir = ['World', 'nes', 'Free Zones', 'Bunkers', 'Areas', 'Special Categories', 'Latin America', 'Europe']

# Crear máscaras para asegurar que ni el socio ni el reportante sean agregados
mascara_partner = ~df_limpio['partnerDesc'].str.contains('|'.join(terminos_excluir), case=False, na=False)
mascara_reporter = ~df_limpio['reporterDesc'].str.contains('|'.join(terminos_excluir), case=False, na=False)

df_limpio = df_limpio[mascara_partner & mascara_reporter]
print(f"3. Filas tras eliminar agregados y territorios no soberanos: {len(df_limpio)}")
# Imprime todos los países reportantes únicos como una lista completa
print(df_limpio['reporterDesc'].unique().tolist())

3. Filas tras eliminar agregados y territorios no soberanos: 141990
['Andorra', 'Angola', 'Azerbaijan', 'Argentina', 'Australia', 'Austria', 'Armenia', 'Barbados', 'Belgium', 'Bermuda', 'Bhutan', 'Bolivia (Plurinational State of)', 'Bosnia Herzegovina', 'Brazil', 'Belize', 'Brunei Darussalam', 'Bulgaria', 'Myanmar', 'Cambodia', 'Canada', 'Sri Lanka', 'Chile', 'China', 'Colombia', 'Comoros', 'Croatia', 'Cyprus', 'Czechia', 'Denmark', 'Dominican Rep.', 'Ecuador', 'El Salvador', 'Estonia', 'Faroe Isds', 'Fiji', 'Finland', 'France', 'Georgia', 'Germany', 'Ghana', 'Greece', 'Guatemala', 'Guyana', 'China, Hong Kong SAR', 'Hungary', 'Iceland', 'Ireland', 'Israel', 'Italy', "Côte d'Ivoire", 'Jamaica', 'Japan', 'Kazakhstan', 'Kenya', 'Rep. of Korea', 'Kuwait', 'Kyrgyzstan', 'Latvia', 'Liberia', 'Lithuania', 'Luxembourg', 'China, Macao SAR', 'Madagascar', 'Malawi', 'Malaysia', 'Maldives', 'Malta', 'Mauritania', 'Mauritius', 'Mexico', 'Rep. of Moldova', 'Montenegro', 'Morocco', 'Mozambique', 'Oma

In [8]:
# 5. Eliminar valores nulos o transacciones en cero en el volumen
# 5. Filtrar registros con volumen comercial válido (primaryValue > 0)
columnas_clave = ['reporterDesc', 'partnerDesc', 'flowDesc', 'cmdCode', 'primaryValue']
df_grafo = df_limpio[df_limpio['primaryValue'] > 0][columnas_clave].copy()
print(f"5. Filas con volumen comercial válido (primaryValue > 0): {len(df_grafo)}")

5. Filas con volumen comercial válido (primaryValue > 0): 141986


In [9]:
# 6. Estandarizar direccionalidad (Origen -> Destino)
df_grafo['Origen'] = df_grafo.apply(
    lambda row: row['partnerDesc'] if row['flowDesc'] == 'Import' else row['reporterDesc'],
    axis=1
)
df_grafo['Destino'] = df_grafo.apply(
    lambda row: row['reporterDesc'] if row['flowDesc'] == 'Import' else row['partnerDesc'],
    axis=1
)

In [10]:
# 7. Agrupar todo el comercio químico bilateral (Manejo de estadísticas espejo)
# Usamos max() en lugar de sum() para evitar duplicar el flujo si ambas partes lo reportan.
# Conservar el registro de mayor denominación mitiga el subreporte de las economías menos rigurosas.
df_enlaces = df_grafo.groupby(['Origen', 'Destino'], as_index=False)['primaryValue'].max()
df_enlaces.rename(columns={'primaryValue': 'Peso_Volumen'}, inplace=True)

In [13]:
# 8. Calcular la resistencia de la ruta
df_enlaces['Resistencia_Ruta'] = 1 / df_enlaces['Peso_Volumen']

print(f"\n¡Éxito! Total de aristas (enlaces) generadas para el grafo: {len(df_enlaces)}")
display(df_enlaces.head())


¡Éxito! Total de aristas (enlaces) generadas para el grafo: 18317


,Origen,Destino,Peso_Volumen,Resistencia_Ruta
0,Afghanistan,Australia,88797.865,0.000011
1,Afghanistan,Austria,32623.743,0.000031
2,Afghanistan,Belgium,2573.130,0.000389
3,Afghanistan,Bolivia (Plurinational State of),20.000,0.050000
4,Afghanistan,Brazil,410159.000,0.000002


In [14]:
df_enlaces.to_csv('Data_Set/Red_Quimica_Comercio.csv', index=False)